# Step 9: RAG Implementation

        _Instructor Solution Notebook_  
        Estimated time: **75 minutes**  
        Difficulty: **Intermediate-Advanced**

        ## Learning Objectives
        - [ ] Load local documents for retrieval.
- [ ] Chunk content into search-friendly segments.
- [ ] Use TF-IDF as a local retrieval baseline.
- [ ] Inject retrieved context into a live model prompt.

        ## Prerequisites
        - Step 8 completed.
- Basic understanding of embeddings and similarity search.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

RAG adds non-parametric context to your system. In this course repo we start with a local TF-IDF retriever so the retrieval mechanics are visible before moving to heavier infrastructure.

Expected output and validation notes:

Expected output snapshot:

- A corpus summary showing document and chunk counts
- Retrieved chunk scores
- A grounded answer that references the retrieved context


## Part 2: Core Implementation

### Index the sample documents


In [ ]:
documents = load_text_documents(resolve_notebook_root() / "data" / "documents")
chunks = chunk_documents(documents)
store = LocalTfidfVectorStore()
store.add_chunks(chunks)

print_json(
    {
        "document_count": len(documents),
        "chunk_count": len(chunks),
    },
    title="RAG corpus summary",
)


## Part 2: Core Implementation

### Retrieve context and ask a live question


In [ ]:
question = "What are the benefits of async programming in Python?"
results = store.search(question, top_k=3)
context = "\n\n".join(
    f"Source: {item['chunk'].title}\n{item['chunk'].content}"
    for item in results
)

rag_agent = create_agent(
    name="RAGAgent",
    instructions="Answer using the supplied context. If the context is thin, say so clearly.",
)
response = await rag_agent.run(
    f"""Context:
    {context}

    Question: {question}
    Answer using the context above."""
)
print(response.text)


## Part 3: Hands-On Exercises

Try a different question and print the retrieval scores before you ask the model.

This mirrored notebook uses completed exercise code so instructors can demonstrate the target state.


In [ ]:
another_question = "How do tools make agent systems more reliable?"
retrieved = store.search(another_question, top_k=3)
for item in retrieved:
    print(item["chunk"].title, round(item["score"], 3))


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
another_question = "How do tools make agent systems more reliable?"
retrieved = store.search(another_question, top_k=3)
for item in retrieved:
    print(item["chunk"].title, round(item["score"], 3))


## Part 5: Best Practices & Tips

        - Inspect retrieval quality before blaming the generation layer.
- Keep chunk sizes and overlap small enough to debug locally.
- Tell the model explicitly how to behave when context is incomplete.


## Summary & Key Takeaways

You built a complete local RAG loop: load, chunk, retrieve, prompt, answer.


## Additional Resources

        - `helpers/rag.py`
- `data/documents`
